## 1 创建python虚拟环境

In [1]:
%pip install -q "uv==0.13.6"
!uv --version
!uv python install 3.12.13
!if [ ! -d .venv ]; then uv venv --managed-python --python 3.12.13 .venv; else echo ".venv 已存在，跳过创建"; fi

uv 0.13.6 (x86_64-unknown-linux-gnu)
Installed Python 3.12.13 in 992ms
 + cpython-3.12.13-linux-x86_64-gnu (python3.12)
Using CPython 3.12.13
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate


In [2]:
%%script .venv/bin/python

import sys
import platform

print("Python 完整版本：", sys.version)
print("Python 路径：", sys.executable)
print("虚拟环境目录：", sys.prefix)
print("基础 Python 目录：", sys.base_prefix)
print("系统架构：", platform.machine())

Python 完整版本： 3.12.13 (main, Aug  7 2026, 02:25:39) [Clang 22.1.3 ]
Python 路径： /content/.venv/bin/python
虚拟环境目录： /content/.venv
基础 Python 目录： /root/.local/share/uv/python/cpython-3.12.13-linux-x86_64-gnu
系统架构： x86_64


In [3]:
!nvidia-smi
!apt-get install -y ffmpeg
!ffmpeg -version | head -n 1
!uv pip install \
    --python /content/.venv/bin/python \
    --torch-backend cu128 \
    "torch==2.11.0+cu128" \
    "torchvision==0.26.0+cu128" \
    "torchcodec==0.11.0+cu128" \
    "numpy==2.0.2" \
    "nvidia-npp-cu12==12.4.1.87"

Sun Nov  1 14:55:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# 在当前 Notebook 内核中配置 NPP 动态库路径，供后续子进程继承。
import os
from pathlib import Path

# 指向虚拟环境内 NPP 的库目录；该路径与 /content/.venv 和 Python 3.12 对应。
npp_dir = Path(
    "/content/.venv/lib/python3.12/site-packages/nvidia/npp/lib"
)

# 检查目标动态库是否存在
libs = list(npp_dir.glob("libnppicc.so*"))
print("NPP 库文件:", libs)

# 未找到目标库时立即报错，提示先检查依赖安装。
assert libs, "NPP 库未找到，请检查安装结果"

# 为后续 %%script 启动的 Python 子进程配置动态库搜索路径
# 保留已有路径；Linux 下多个动态库搜索目录使用冒号分隔。
old_path = os.environ.get("LD_LIBRARY_PATH", "")
paths = old_path.split(":") if old_path else []

# 将 NPP 目录放在搜索路径前面，并避免重复运行时重复添加。
if str(npp_dir) not in paths:
    os.environ["LD_LIBRARY_PATH"] = (
        str(npp_dir) + (":" + old_path if old_path else "")
    )

print("NPP 动态库路径配置成功")

NPP 库文件: [PosixPath('/content/.venv/lib/python3.12/site-packages/nvidia/npp/lib/libnppicc.so.12')]
NPP 动态库路径配置成功


In [5]:
%%script /content/.venv/bin/python

# 在虚拟环境子进程中检查依赖版本，并执行一次 GPU 矩阵乘法。
# 先运行单元格 4，使此子进程继承配置好的 NPP 动态库搜索路径。
import torch
import torchvision
import numpy as np
import torchcodec

# 打印实际导入的库版本，便于核对安装结果。
print("PyTorch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("TorchCodec:", torchcodec.__version__)
print("NumPy:", np.__version__)

# 这里的 CUDA 版本是 PyTorch 构建所用的版本；is_available() 检查 CUDA 是否可用。
print("CUDA:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())

# 在 GPU 上创建 1024×1024 的随机矩阵并相乘；CUDA 不可用时这里会报错。
x = torch.randn(1024, 1024, device="cuda")
y = x @ x

# 输出结果形状，用于基本冒烟检查；此处没有显式同步 GPU 或校验数值精度。
print("GPU 计算测试通过:", y.shape)

PyTorch: 2.11.0+cu128
torchvision: 0.26.0+cu128
TorchCodec: 0.11.0+cu128
NumPy: 2.0.2
CUDA: 12.8
GPU available: True
GPU 计算测试通过: torch.Size([1024, 1024])


## 2.下载Lerobot源码安装

In [6]:
!git clone --branch v0.6.1 --depth 1 https://github.com/huggingface/lerobot.git /content/lerobot
!git -C /content/lerobot describe --tags --exact-match
!git -C /content/lerobot rev-parse HEAD

Cloning into '/content/lerobot'...
remote: Enumerating objects: 1187, done.
remote: Counting objects: 100% (1187/1187), done.
remote: Compressing objects: 100% (1041/1041), done.
remote: Total 1187 (delta 153), reused 813 (delta 107), pack-reused 0 (from 0)
Receiving objects: 100% (1187/1187), 6.35 MiB | 33.89 MiB/s, done.
Resolving deltas: 100% (153/153), done.
Note: switching to '7e241bd630a3719a56157a497ce5d08f244784f1'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

v0.6.1
7e241bd630a3719a56157a497ce5d08f244784f1


In [7]:
!uv pip install \
    --python /content/.venv/bin/python \
    --torch-backend cu128 \
    -e "/content/lerobot[smolvla,evaluation,libero]" \
    "torch==2.11.0+cu128" \
    "torchvision==0.26.0+cu128" \
    "torchcodec==0.11.0+cu128" \
    "numpy==2.0.2" \
    "nvidia-npp-cu12==12.4.1.87"

!uv pip check --python /content/.venv/bin/python

Resolved 165 packages in 1.97s                                       
Prepared 133 packages in 7.61s                                           
Uninstalled 2 packages in 3ms
Installed 133 packages in 131ms                             
 + absl-py==2.5.1
 + accelerate==1.15.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + antlr4-python3-runtime==4.9.3
 + anyio==4.16.0
 + attrs==26.1.0
 + av==15.1.0
 + bddl==1.0.1
 + certifi==2026.10.23
 + charset-normalizer==3.5.2
 + click==8.5.0
 + cloudpickle==3.1.2
 + cmake==4.1.3
 + contourpy==1.4.0
 + cwevals==0.0.1b1
 + cycler==0.12.1
 + datasets==4.8.5
 + dill==0.4.1
 + distro==1.9.0
 + docopt==0.6.2
 + draccus==0.11.6
 + easydict==1.13
 + egl-probe==1.0.2
 + einops==0.8.2
 + etils==1.14.0
 + farama-notifications==0.0.6
 + fastjsonschema==2.22.2
 + fonttools==4.68.0
 + frozenlist==1.8.0
 - fsspec==2026.9.0
 + fsspec==2026.2.0
 + future==1.0.0
 + glfw==2.11.0
 + googleapis-comm

## 3.登录HuggingFace

In [8]:
from huggingface_hub import login
login()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


## 4.下载libero仿真资源

In [9]:
%%script /content/.venv/bin/python

from pathlib import Path
from importlib.util import find_spec
import yaml

# 1. 定位 LIBERO 安装目录（不触发 import）
spec = find_spec("libero")
assert spec is not None and spec.origin is not None

libero_root = Path(spec.origin).resolve().parent / "libero"

# 2. 创建 LIBERO 配置目录
config_dir = Path.home() / ".libero"
config_dir.mkdir(parents=True, exist_ok=True)

# 3. 配置仿真资源和数据目录
assets_dir = Path.home() / ".cache/libero/assets"
datasets_dir = config_dir / "datasets"
datasets_dir.mkdir(parents=True, exist_ok=True)

# 4. 写入 LIBERO 配置文件
config = {
    "benchmark_root": str(libero_root),
    "assets": str(assets_dir),
    "bddl_files": str(libero_root / "bddl_files"),
    "init_states": str(libero_root / "init_files"),
    "datasets": str(datasets_dir),
}

config_path = config_dir / "config.yaml"

with config_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False)

# 5. 验证配置
assert (libero_root / "bddl_files").is_dir()
assert (libero_root / "init_files").is_dir()

print("LIBERO 安装目录:", libero_root)
print("Assets 缓存目录:", assets_dir)
print("配置文件:", config_path)
print("\nLIBERO 配置初始化完成")

LIBERO 安装目录: /content/.venv/lib/python3.12/site-packages/libero/libero
Assets 缓存目录: /root/.cache/libero/assets
配置文件: /root/.libero/config.yaml

LIBERO 配置初始化完成


In [10]:
%%script /content/.venv/bin/python

from pathlib import Path
from huggingface_hub import snapshot_download

# 1. 指定 LIBERO 默认的 Assets 缓存目录
assets_dir = Path.home() / ".cache/libero/assets"

# 2. 从 Hugging Face 下载完整仿真资源
snapshot_download(
    repo_id="lerobot/libero-assets",
    repo_type="dataset",
    local_dir=str(assets_dir),
)

# 3. 检查资源
assert (assets_dir / "scenes").is_dir(), "场景资源缺失"
assert (assets_dir / "textures").is_dir(), "纹理资源缺失"

print("LIBERO Assets 下载完成")
print("资源目录:", assets_dir)
print("文件数量:", sum(1 for p in assets_dir.rglob("*") if p.is_file()))

LIBERO Assets 下载完成
资源目录: /root/.cache/libero/assets
文件数量: 1761


Fetching 586 files: 100%|██████████| 586/586 [00:13<00:00, 43.16it/s]


## 5.加载模型

In [11]:
%%script /content/.venv/bin/python

import torch
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

# 加载 LIBERO 微调后的模型
repo_id = "lemons2027/smolvla-libero-baseline-20k"

policy = SmolVLAPolicy.from_pretrained(repo_id)
policy = policy.to("cuda").eval()

print("模型加载成功！")

# 检查模型配置
cfg = policy.config

print("\n=== 模型输入 ===")
for name, feature in cfg.input_features.items():
    print(name, feature.shape)

print("\n=== 模型输出 ===")
for name, feature in cfg.output_features.items():
    print(name, feature.shape)

print("\n=== 动作预测 ===")
print("Chunk Size:", cfg.chunk_size)
print("Action Steps:", cfg.n_action_steps)
print("Flow Matching Steps:", cfg.num_steps)

print("\n=== GPU ===")
print("GPU:", torch.cuda.get_device_name(0))
print("显存占用:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GiB")

Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...
Reducing the number of VLM layers to 16 ...
模型加载成功！

=== 模型输入 ===
observation.images.image (3, 256, 256)
observation.images.image2 (3, 256, 256)
observation.state (8,)

=== 模型输出 ===
action (7,)

=== 动作预测 ===
Chunk Size: 50
Action Steps: 50
Flow Matching Steps: 10

=== GPU ===
GPU: Tesla T4
显存占用: 0.86 GiB


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 489/489 [00:00<00:00, 2337.45it/s]


## 6.闭环测试

### 6.1 Spatial-Object-Goal-Long 套件-Task 0~9-Init 0~9

In [13]:
import os
import json
import shutil
import subprocess
from pathlib import Path
from google.colab import drive

# ============================================================
# 1. 挂载 Google Drive
# ============================================================

drive.mount("/content/drive")

# ============================================================
# 2. 实验配置
# ============================================================

repo_id = "lemons2027/smolvla-libero-baseline-20k"

local_root = Path("/content/outputs/smolvla_libero_eval_400")
drive_root = Path("/content/drive/MyDrive/SmolVLA/rollout/baseline_20k")

local_root.mkdir(parents=True, exist_ok=True)
drive_root.mkdir(parents=True, exist_ok=True)

# 四个任务套件，每个套件 10 Tasks × 10 Episodes
suites = [
    "libero_spatial",
    "libero_object",
    "libero_goal",
    "libero_10",
]

# ============================================================
# 3. 配置子进程运行环境
# ============================================================

env = os.environ.copy()

env["MUJOCO_GL"] = "egl"
env["MPLBACKEND"] = "Agg"

# 减少支持此环境变量的进度条输出
env["TQDM_DISABLE"] = "1"
env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

# TorchCodec 所需的 NPP 动态库
npp_dir = Path(
    "/content/.venv/lib/python3.12/site-packages/nvidia/npp/lib"
)

assert list(npp_dir.glob("libnppicc.so.12*")), "NPP 动态库缺失"

paths = env.get("LD_LIBRARY_PATH", "").split(":")

if str(npp_dir) not in paths:
    old_path = env.get("LD_LIBRARY_PATH", "")
    env["LD_LIBRARY_PATH"] = (
        str(npp_dir) + (":" + old_path if old_path else "")
    )

# ============================================================
# 4. 检查套件评测结果是否完整
# ============================================================

def is_complete(directory, suite):
    result_path = directory / "eval_info.json"

    if not result_path.is_file():
        return False

    try:
        with result_path.open("r", encoding="utf-8") as f:
            results = json.load(f)

        tasks = sorted(
            [
                t for t in results["per_task"]
                if t["task_group"] == suite
            ],
            key=lambda t: t["task_id"]
        )

        # 必须包含 Task 0～9
        if [t["task_id"] for t in tasks] != list(range(10)):
            return False

        # 每个 Task 必须有10个Episode和10个视频
        for task in tasks:
            task_id = task["task_id"]
            metrics = task["metrics"]

            if len(metrics["successes"]) != 10:
                return False

            video_paths = metrics.get("video_paths", [])

            if len(video_paths) != 10:
                return False

            # 检查实际视频是否存在
            for video in video_paths:
                video_file = (
                    directory
                    / "videos"
                    / f"{suite}_{task_id}"
                    / Path(video).name
                )

                if not video_file.is_file():
                    return False

                if video_file.stat().st_size == 0:
                    return False

        return results["overall"]["n_episodes"] == 100

    except (KeyError, TypeError, ValueError, OSError):
        return False


# ============================================================
# 5. 顺序执行四个套件
# ============================================================

print("=" * 60)
print("SmolVLA LIBERO Baseline Rollout")
print("模型:", repo_id)
print("任务套件: Spatial / Object / Goal / Long")
print("Task: 40")
print("Episodes: 400")
print("控制频率: 20 Hz")
print("Batch Size: 1")
print("Seed: 42")
print("=" * 60, flush=True)

for index, suite in enumerate(suites, start=1):

    output_dir = local_root / suite
    backup_dir = drive_root / suite
    log_path = local_root / f"{suite}.log"
    marker = backup_dir / ".backup_complete"

    print(f"\n[{index}/4] {suite}", flush=True)

    # 已完成且备份成功，则直接跳过
    if marker.is_file() and is_complete(backup_dir, suite):
        with (backup_dir / "eval_info.json").open(
            "r", encoding="utf-8"
        ) as f:
            results = json.load(f)

        overall = results["overall"]

        print(
            f"Drive 中已有完整结果，跳过评测 | "
            f"Episodes: {overall['n_episodes']} | "
            f"Success Rate: {overall['pc_success']:.2f}%",
            flush=True
        )
        continue

    # 本地尚未完成，则执行该套件评测
    if not is_complete(output_dir, suite):

        # 防止覆盖不完整的历史评测目录
        if output_dir.exists() and any(output_dir.iterdir()):
            raise RuntimeError(
                f"检测到未完成的评测目录：{output_dir}\n"
                "请先检查该目录，再决定是否重新运行。"
            )

        cmd = [
            "/content/.venv/bin/python",
            "-m", "lerobot.scripts.lerobot_eval",

            f"--policy.path={repo_id}",
            "--policy.device=cuda",

            "--env.type=libero",
            f"--env.task={suite}",
            "--env.fps=20",
            "--env.control_mode=relative",
            "--env.init_states=true",
            "--env.observation_height=256",
            "--env.observation_width=256",
            "--env.max_parallel_tasks=1",

            "--eval.batch_size=1",
            "--eval.n_episodes=10",
            "--seed=42",

            f"--output_dir={output_dir}",
        ]

        print("开始：10 Tasks × 10 Episodes", flush=True)
        print("完整日志:", log_path, flush=True)

        # 训练/评测终端输出全部写入文件，不刷 Notebook
        with log_path.open("w", encoding="utf-8") as log_file:
            result = subprocess.run(
                cmd,
                env=env,
                stdout=log_file,
                stderr=subprocess.STDOUT,
                check=False,
            )

        if result.returncode != 0:
            # 失败时也保存日志
            shutil.copy2(
                log_path,
                drive_root / f"{suite}_failed.log"
            )

            print("\n评测失败，最后30行日志：")
            print("\n".join(
                log_path.read_text(
                    encoding="utf-8",
                    errors="replace"
                ).splitlines()[-30:]
            ))

            raise RuntimeError(
                f"{suite} Rollout 失败，已停止后续评测"
            )

        assert is_complete(output_dir, suite), (
            f"{suite} 评测程序已结束，但结果不完整"
        )

    # ========================================================
    # 6. 保存评测协议与日志
    # ========================================================

    protocol = {
        "policy": repo_id,
        "lerobot_version": "v0.6.1",
        "suite": suite,
        "task_ids": list(range(10)),
        "episodes_per_task": 10,
        "control_frequency": 20,
        "control_mode": "relative",
        "init_states": True,
        "eval_batch_size": 1,
        "max_parallel_tasks": 1,
        "observation_height": 256,
        "observation_width": 256,
        "seed": 42,
    }

    (output_dir / "protocol.json").write_text(
        json.dumps(protocol, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

    if log_path.exists():
        shutil.copy2(log_path, output_dir / "rollout.log")

    # ========================================================
    # 7. 备份到 Google Drive
    # ========================================================

    print("评测完成，正在备份...", flush=True)

    # 只有全部复制验证完才重新标记为完成
    marker.unlink(missing_ok=True)

    shutil.copytree(
        output_dir,
        backup_dir,
        dirs_exist_ok=True
    )

    # 比较全部文件的相对路径及大小
    local_files = {
        p.relative_to(output_dir): p.stat().st_size
        for p in output_dir.rglob("*")
        if p.is_file()
    }

    backup_files = {
        p.relative_to(backup_dir): p.stat().st_size
        for p in backup_dir.rglob("*")
        if p.is_file() and p.name != ".backup_complete"
    }

    assert local_files == backup_files, (
        f"{suite} Google Drive 备份文件不完整"
    )

    assert is_complete(backup_dir, suite), (
        f"{suite} Google Drive 评测结果校验失败"
    )

    # 最后才写完成标记
    marker.write_text("verified", encoding="utf-8")

    # ========================================================
    # 8. 输出本套件的简洁结果
    # ========================================================

    with (backup_dir / "eval_info.json").open(
        "r", encoding="utf-8"
    ) as f:
        results = json.load(f)

    overall = results["overall"]

    print(
        f"完成：{suite} | "
        f"Episodes: {overall['n_episodes']} | "
        f"Success Rate: {overall['pc_success']:.2f}%",
        flush=True
    )

    print("Google Drive 备份完成", flush=True)


print("\n四个任务套件处理结束。")
print("评测输出目录:", local_root)
print("Google Drive 目录:", drive_root)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SmolVLA LIBERO Baseline Rollout
模型: lemons2027/smolvla-libero-baseline-20k
任务套件: Spatial / Object / Goal / Long
Task: 40
Episodes: 400
控制频率: 20 Hz
Batch Size: 1
Seed: 42

[1/4] libero_spatial
Drive 中已有完整结果，跳过评测 | Episodes: 100 | Success Rate: 65.00%

[2/4] libero_object
Drive 中已有完整结果，跳过评测 | Episodes: 100 | Success Rate: 62.00%

[3/4] libero_goal
Drive 中已有完整结果，跳过评测 | Episodes: 100 | Success Rate: 73.00%

[4/4] libero_10
Drive 中已有完整结果，跳过评测 | Episodes: 100 | Success Rate: 46.00%

四个任务套件处理结束。
评测输出目录: /content/outputs/smolvla_libero_eval_400
Google Drive 目录: /content/drive/MyDrive/SmolVLA/rollout/baseline_20k
